In [130]:
import pandas as pd
import numpy as np
from glob import glob
import time
import getpass
import dill
import os

import arcgis
from arcgis.features import SpatialDataFrame
from arcgis.features.use_proximity import find_nearest

In [15]:
portals_dict = {
    "esrifederal_gis": r"https://esrifederal.maps.arcgis.com",
    "natgov_gis": r"http://esri-natgov105.eastus.cloudapp.azure.com/arcgis",
    "dot_gis": r"http://dot.esri.com/portal",
    "dev_gis": r"http://govdev.eastus.cloudapp.azure.com/arcgis",
    "local_gis": r"https://anieto.esri.com/arcgis",
    "science_agol_org": r"https://science.maps.arcgis.com"
}

In [16]:
gis_url = portals_dict["esrifederal_gis"]

In [17]:
if gis_url == portals_dict["esrifederal_gis"]:
#     gis_app_id = getpass.getpass(prompt="App ID: ")
    gis_app_id = r"wt3QUR1M4eum0TVI"
    print("Attempting to log in to '{0}'...".format(gis_url))
    gis = arcgis.gis.GIS(gis_url, client_id=gis_app_id)
    print("Successfully logged in as: " + gis.properties.user.username)
else:
    gis_username = getpass.getpass(prompt="Username: ")
    gis_pw = getpass.getpass(prompt="Password: ")
    print("Attempting to log in to '{0}'...".format(gis_url))
    gis = arcgis.gis.GIS(gis_url, gis_username, gis_pw, verify_cert=False)
    print("Successfully logged in as: " + gis.properties.user.username)

Attempting to log in to 'https://esrifederal.maps.arcgis.com'...
Please sign in to your GIS and paste the code that is obtained below.
If a web browser does not automatically open, please navigate to the URL below yourself instead.
Opening web browser to navigate to: https://esrifederal.maps.arcgis.com/sharing/rest/oauth2/authorize?redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&client_id=wt3QUR1M4eum0TVI&expiration=-1&response_type=code
Enter code obtained on signing in using SAML: ········
Successfully logged in as: albe9057@esri.com_esrifederal


In [18]:
root = r'D:\GitHub\zika' 

## Places with confirmed Zika infections

Uses CDC Epidemic Prediction Initiative's repository of publicly available Zika data - https://github.com/cdcepi/zika

In [ ]:
location_files = glob(root +'/*/*Places.csv')

In [ ]:
locations = pd.concat([pd.read_csv(x, encoding="ISO-8859-1")
                       for x in location_files], axis=0).reset_index(drop=True)

In [ ]:
locations.head(1)

In [ ]:
# clean up
locations.fillna('', inplace=True)
locations['country'] = locations.country.str.replace('_', ' ')

cols = locations.select_dtypes(include=[np.object]).columns
locations[cols] = locations[cols].apply(
    lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))

### Create address column

In [ ]:
def get_address(location):
    addr_parts = location.split('-')
    addr = addr_parts[-1].replace('_', ' ')
    for part in addr_parts[-2::-1]:
        addr += ', ' + part.replace('_', ' ')
    return addr

In [ ]:
locations['address'] = locations['location'].apply(get_address)

In [ ]:
locations.head(1)

### Create spatial dataframe

Create a spatially enabled dataframe, save it as a shapefile, and also import it into the GIS

In [ ]:
spdf = SpatialDataFrame.from_df(locations)

In [ ]:
spdf.sr = arcgis.geometry.SpatialReference(wkid=4326)

In [ ]:
spdf.head()

In [ ]:
spdf.to_featureclass(out_location=root,
                         out_name="zika_locations.shp")

In [ ]:
zika_loc = gis.content.import_data(spdf, title='Locations with Zika')

In [ ]:
zika_loc

In [ ]:
zika_loc.itemid

## Load Zika locations feature layer

In [ ]:
zika_loc = gis.content.get('64032f4bee864dd093c74f839dcc1d87')

In [ ]:
zikalyr = zika_loc.layers[0]

### Map locations of zika infections

In [ ]:
m = gis.map('USA')
m

In [ ]:
m.add_layer(zika_loc)

## Find airports nearest to Zika locations

### Load Airport locations

In [ ]:
airports = gis.content.search('title: World Airport Locations owner:garycjohnsonDP', 'Feature Layer', outside_org=True)[0]
airports

In [ ]:
airportlyr = airports.layers[0]

In [ ]:
airportsdf = SpatialDataFrame.from_layer(airportlyr)
airportsdf

In [ ]:
airportsdf.shape

In [ ]:
zikadf = zikalyr.query().df
zikadf

In [ ]:
zikadf.shape

In [ ]:
zikadf.country.unique()

In [ ]:
zikadf.columns

### Find Nearest Airports from Zika locations using Spatial Analysis

In [ ]:
def find_nearest_airports(location_filter, airport_filter):
    zikalyr.filter = location_filter
    airportlyr.filter = airport_filter
    
    nearest_airports = find_nearest(zikalyr, airportlyr, 'StraightLine', max_count=1,
                                    search_cutoff=100, search_cutoff_units='Miles')

    m.add_layer(nearest_airports['nearest_layer'])
    m.add_layer(nearest_airports['connecting_lines_layer'])
    
    nearestdf = nearest_airports['connecting_lines_layer'].query().df
    return nearestdf[['From_ID', 'From_Name', 'OID', 'To_Airport_Name', 'To_IATA', 'To_ICAO', 'To_ID', 'Total_Miles']]

In [ ]:
location_airport_df = pd.DataFrame()

In [ ]:
m

In [ ]:
for country in zikadf.country.unique():
    print('Finding nearest airports for locations in: ' + country)
    try:
        location_filter = "Country = '{}'".format(country)
        airport_filter = "Country = '{}'".format(country.upper())

        if country == 'United States':
            airport_filter = "COUNTRY='USA' OR COUNTRY='PUERTO RICO'"

        df = find_nearest_airports(location_filter, airport_filter)

        location_airport_df = location_airport_df.append(df, ignore_index=True)
    except:
        print('Unable to find nearest airports for ' + country)

In [ ]:
# Determine the amount of cases in Colombia
zikadf.loc[zikadf['country'] == "Colombia"].shape

In [ ]:
colombiadf = zikadf.loc[zikadf['country'] == "Colombia"]
colombiadf

In [ ]:
zikalyr.properties

In [ ]:
airport_filter  = "Country = 'Colombia'"

location_filter = "Country = 'Colombia' AND OBJECTID < 1000"
df = find_nearest_airports(location_filter, airport_filter)
location_airport_df = location_airport_df.append(df, ignore_index=True)

location_filter = "Country = 'Colombia' AND OBJECTID >= 1000"
df = find_nearest_airports(location_filter, airport_filter)
location_airport_df = location_airport_df.append(df, ignore_index=True)

In [ ]:
location_airport_df.head()

In [ ]:
loc_apt = root + '\\location_airport_df.pkl'
location_airport_df.to_pickle(loc_apt)

## Load nearest airports

### Resume from here if pkl files from previous cells already available


In [19]:
def get_address(location):
    addr_parts = location.split('-')
    addr = addr_parts[-1].replace('_', ' ')
    for part in addr_parts[-2::-1]:
        addr += ', ' + part.replace('_', ' ')
    return addr

In [20]:
loc_apt = root + '\\location_airport_df.pkl'
location_airport_df = pd.read_pickle(loc_apt)

In [21]:
location_airport_df.head()

,From_ID,From_Name,OID,To_Airport_Name,To_IATA,To_ICAO,To_ID,Total_Miles
0,1,"Buenos Aires, Argentina",1,AEROPARQUE JORGE NEWBERY,AEP,SABE,6002,4.153021
1,2,"CABA, Argentina",2,AEROPARQUE JORGE NEWBERY,AEP,SABE,6002,4.153021
2,3,"Cordoba, Argentina",3,AMBROSIO L V TARAVELLA,COR,SACO,6019,5.890400
3,4,"Entre Rios, Argentina",4,GUALEGUAYCHU,GHU,SAAG,5972,73.439201
4,5,"Santa Fe, Argentina",5,SAUCE VIEJO,SFN,SAAV,5998,7.804494


## Load infection data

In [22]:
data_file_locations = glob(root + '/*/*/data/*.csv')
data = pd.concat([pd.read_csv(x, encoding="ISO-8859-1")
                            for x in data_file_locations], axis=0).reset_index(drop=True)


In [23]:
data

,Unnamed: 9,data_field,data_field_code,location,location_type,report_date,time_period,time_period_type,unit,value
0,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Buenos_Aires,province,2017-01-12,NaN,NaN,cases,0.0
1,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-CABA,province,2017-01-12,NaN,NaN,cases,1.0
2,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Cordoba,province,2017-01-12,NaN,NaN,cases,2.0
3,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Entre_Rios,province,2017-01-12,NaN,NaN,cases,0.0
4,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Santa_Fe,province,2017-01-12,NaN,NaN,cases,2.0
5,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Mendoza,province,2017-01-12,NaN,NaN,cases,1.0
6,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-San_Juan,province,2017-01-12,NaN,NaN,cases,0.0
7,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-San_Luis,province,2017-01-12,NaN,NaN,cases,0.0
8,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Chaco,province,2017-01-12,NaN,NaN,cases,0.0
9,NaN,cumulative_confirmed_imported_cases,AR0003,Argentina-Corrientes,province,2017-01-12,NaN,NaN,cases,0.0


In [24]:
data.drop(['Unnamed: 9', 'time_period','time_period_type'], axis=1, inplace=True)

cols = data.select_dtypes(include=[np.object]).columns
data[cols] = data[cols].apply(
    lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))

In [25]:
# clean up - some columns are off 
data.dropna(subset=['unit'], inplace=True)

In [26]:
data

,data_field,data_field_code,location,location_type,report_date,unit,value
0,cumulative_confirmed_imported_cases,AR0003,Argentina-Buenos_Aires,province,2017-01-12,cases,0.0
1,cumulative_confirmed_imported_cases,AR0003,Argentina-CABA,province,2017-01-12,cases,1.0
2,cumulative_confirmed_imported_cases,AR0003,Argentina-Cordoba,province,2017-01-12,cases,2.0
3,cumulative_confirmed_imported_cases,AR0003,Argentina-Entre_Rios,province,2017-01-12,cases,0.0
4,cumulative_confirmed_imported_cases,AR0003,Argentina-Santa_Fe,province,2017-01-12,cases,2.0
5,cumulative_confirmed_imported_cases,AR0003,Argentina-Mendoza,province,2017-01-12,cases,1.0
6,cumulative_confirmed_imported_cases,AR0003,Argentina-San_Juan,province,2017-01-12,cases,0.0
7,cumulative_confirmed_imported_cases,AR0003,Argentina-San_Luis,province,2017-01-12,cases,0.0
8,cumulative_confirmed_imported_cases,AR0003,Argentina-Chaco,province,2017-01-12,cases,0.0
9,cumulative_confirmed_imported_cases,AR0003,Argentina-Corrientes,province,2017-01-12,cases,0.0


In [27]:
data = data.loc[data.report_date != '18437']

In [28]:
# data['report_date'].fillna(data['ï»¿report_date'], inplace=True)
# del data['ï»¿report_date']

data['report_date'] = data.report_date.str.replace('_','-')       
data['report_date'] = pd.to_datetime(data.report_date)

C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\lib\site-packages\ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  after removing the cwd from sys.path.
C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """


In [29]:
data.report_date.unique()

array(['2017-01-12T00:00:00.000000000', '2017-01-19T00:00:00.000000000',
       '2017-02-17T00:00:00.000000000', '2017-02-24T00:00:00.000000000',
       '2017-03-06T00:00:00.000000000', '2017-03-13T00:00:00.000000000',
       '2017-03-20T00:00:00.000000000', '2017-03-27T00:00:00.000000000',
       '2016-03-28T00:00:00.000000000', '2017-04-03T00:00:00.000000000',
       '2016-04-02T00:00:00.000000000', '2017-04-10T00:00:00.000000000',
       '2016-04-08T00:00:00.000000000', '2016-04-16T00:00:00.000000000',
       '2016-04-22T00:00:00.000000000', '2017-04-24T00:00:00.000000000',
       '2016-04-29T00:00:00.000000000', '2016-05-07T00:00:00.000000000',
       '2017-05-09T00:00:00.000000000', '2016-05-14T00:00:00.000000000',
       '2016-05-22T00:00:00.000000000', '2017-05-22T00:00:00.000000000',
       '2016-05-30T00:00:00.000000000', '2016-06-06T00:00:00.000000000',
       '2017-06-05T00:00:00.000000000', '2016-06-13T00:00:00.000000000',
       '2016-06-18T00:00:00.000000000', '2017-06-19

In [30]:
data.head()

,data_field,data_field_code,location,location_type,report_date,unit,value
0,cumulative_confirmed_imported_cases,AR0003,Argentina-Buenos_Aires,province,2017-01-12,cases,0.0
1,cumulative_confirmed_imported_cases,AR0003,Argentina-CABA,province,2017-01-12,cases,1.0
2,cumulative_confirmed_imported_cases,AR0003,Argentina-Cordoba,province,2017-01-12,cases,2.0
3,cumulative_confirmed_imported_cases,AR0003,Argentina-Entre_Rios,province,2017-01-12,cases,0.0
4,cumulative_confirmed_imported_cases,AR0003,Argentina-Santa_Fe,province,2017-01-12,cases,2.0


In [31]:
# Drop municipalities
data = data.loc[data.unit!='municipalities']
data = (data[['report_date', 'location', 'value', 'data_field']]
             .rename(columns={'report_date':'date','value':'zika_cases'}))

In [32]:
data['zika_cases'] = data.zika_cases.fillna(0)
data['zika_cases'] = data.zika_cases.astype(int)

In [33]:
data.head()

,date,location,zika_cases,data_field
0,2017-01-12,Argentina-Buenos_Aires,0,cumulative_confirmed_imported_cases
1,2017-01-12,Argentina-CABA,1,cumulative_confirmed_imported_cases
2,2017-01-12,Argentina-Cordoba,2,cumulative_confirmed_imported_cases
3,2017-01-12,Argentina-Entre_Rios,0,cumulative_confirmed_imported_cases
4,2017-01-12,Argentina-Santa_Fe,2,cumulative_confirmed_imported_cases


In [34]:
data.query("zika_cases>0").shape, data.shape

((94721, 4), (232902, 4))

In [35]:
# Remove data that doesn't appear to be directly associated with Zika
excluded_fields = ['cumulative_cases_discarded',
'microcephaly_not',
'gbs_reported',
'zika_not',
'confirmed_acute_fever',
'confirmed_arthralgia',
'confirmed_arthritis', 
'confirmed_rash', 
'confirmed_conjunctivitis',
'confirmed_eyepain', 
'confirmed_headache', 
'confirmed_malaise',
'zika_reported_travel',
'yearly_reported_travel_cases']

mask = data.data_field.isin(excluded_fields)
print(mask.sum(), data.loc[mask, 'zika_cases'].sum(), data.zika_cases.sum())

6667 611942 18605000


In [36]:
data = data.loc[mask.pipe(np.invert)]

In [37]:
data.shape[0], data.zika_cases.sum()

(226235, 17993058)

In [38]:
data.dtypes

date          datetime64[ns]
location              object
zika_cases             int32
data_field            object
dtype: object

In [39]:
data_pkl = root + '\\infection_data_final.pkl'
data.to_pickle(data_pkl)

In [40]:
infection_data = pd.read_pickle(root + '\\infection_data_final.pkl')

In [41]:
infection_data.head()

,date,location,zika_cases,data_field
0,2017-01-12,Argentina-Buenos_Aires,0,cumulative_confirmed_imported_cases
1,2017-01-12,Argentina-CABA,1,cumulative_confirmed_imported_cases
2,2017-01-12,Argentina-Cordoba,2,cumulative_confirmed_imported_cases
3,2017-01-12,Argentina-Entre_Rios,0,cumulative_confirmed_imported_cases
4,2017-01-12,Argentina-Santa_Fe,2,cumulative_confirmed_imported_cases


In [42]:
infection_data['address'] = infection_data['location'].apply(get_address)
infection_data

,date,location,zika_cases,data_field,address
0,2017-01-12,Argentina-Buenos_Aires,0,cumulative_confirmed_imported_cases,"Buenos Aires, Argentina"
1,2017-01-12,Argentina-CABA,1,cumulative_confirmed_imported_cases,"CABA, Argentina"
2,2017-01-12,Argentina-Cordoba,2,cumulative_confirmed_imported_cases,"Cordoba, Argentina"
3,2017-01-12,Argentina-Entre_Rios,0,cumulative_confirmed_imported_cases,"Entre Rios, Argentina"
4,2017-01-12,Argentina-Santa_Fe,2,cumulative_confirmed_imported_cases,"Santa Fe, Argentina"
5,2017-01-12,Argentina-Mendoza,1,cumulative_confirmed_imported_cases,"Mendoza, Argentina"
6,2017-01-12,Argentina-San_Juan,0,cumulative_confirmed_imported_cases,"San Juan, Argentina"
7,2017-01-12,Argentina-San_Luis,0,cumulative_confirmed_imported_cases,"San Luis, Argentina"
8,2017-01-12,Argentina-Chaco,0,cumulative_confirmed_imported_cases,"Chaco, Argentina"
9,2017-01-12,Argentina-Corrientes,0,cumulative_confirmed_imported_cases,"Corrientes, Argentina"


infection_data = infection_data[['date','address']]
infection_data.head(1)

In [43]:
location_airport_df.head()

,From_ID,From_Name,OID,To_Airport_Name,To_IATA,To_ICAO,To_ID,Total_Miles
0,1,"Buenos Aires, Argentina",1,AEROPARQUE JORGE NEWBERY,AEP,SABE,6002,4.153021
1,2,"CABA, Argentina",2,AEROPARQUE JORGE NEWBERY,AEP,SABE,6002,4.153021
2,3,"Cordoba, Argentina",3,AMBROSIO L V TARAVELLA,COR,SACO,6019,5.890400
3,4,"Entre Rios, Argentina",4,GUALEGUAYCHU,GHU,SAAG,5972,73.439201
4,5,"Santa Fe, Argentina",5,SAUCE VIEJO,SFN,SAAV,5998,7.804494


## Combine infection date with nearest airport

In [44]:
merge_all = pd.merge(infection_data, 
                     location_airport_df[['From_Name', 'To_Airport_Name', 'To_IATA', 'To_ICAO']], 
                     left_on='address', 
                     right_on='From_Name',
                     how='left').drop_duplicates()

In [45]:
merge_all.head()

,date,location,zika_cases,data_field,address,From_Name,To_Airport_Name,To_IATA,To_ICAO
0,2017-01-12,Argentina-Buenos_Aires,0,cumulative_confirmed_imported_cases,"Buenos Aires, Argentina","Buenos Aires, Argentina",AEROPARQUE JORGE NEWBERY,AEP,SABE
1,2017-01-12,Argentina-CABA,1,cumulative_confirmed_imported_cases,"CABA, Argentina","CABA, Argentina",AEROPARQUE JORGE NEWBERY,AEP,SABE
2,2017-01-12,Argentina-Cordoba,2,cumulative_confirmed_imported_cases,"Cordoba, Argentina","Cordoba, Argentina",AMBROSIO L V TARAVELLA,COR,SACO
3,2017-01-12,Argentina-Entre_Rios,0,cumulative_confirmed_imported_cases,"Entre Rios, Argentina","Entre Rios, Argentina",GUALEGUAYCHU,GHU,SAAG
4,2017-01-12,Argentina-Santa_Fe,2,cumulative_confirmed_imported_cases,"Santa Fe, Argentina","Santa Fe, Argentina",SAUCE VIEJO,SFN,SAAV


In [46]:
merge_all.shape

(226035, 9)

In [47]:
merge_all.rename(columns={'To_IATA':'IATA','To_ICAO':'ICAO'}, inplace=True)

In [48]:
merge_all.head()

,date,location,zika_cases,data_field,address,From_Name,To_Airport_Name,IATA,ICAO
0,2017-01-12,Argentina-Buenos_Aires,0,cumulative_confirmed_imported_cases,"Buenos Aires, Argentina","Buenos Aires, Argentina",AEROPARQUE JORGE NEWBERY,AEP,SABE
1,2017-01-12,Argentina-CABA,1,cumulative_confirmed_imported_cases,"CABA, Argentina","CABA, Argentina",AEROPARQUE JORGE NEWBERY,AEP,SABE
2,2017-01-12,Argentina-Cordoba,2,cumulative_confirmed_imported_cases,"Cordoba, Argentina","Cordoba, Argentina",AMBROSIO L V TARAVELLA,COR,SACO
3,2017-01-12,Argentina-Entre_Rios,0,cumulative_confirmed_imported_cases,"Entre Rios, Argentina","Entre Rios, Argentina",GUALEGUAYCHU,GHU,SAAG
4,2017-01-12,Argentina-Santa_Fe,2,cumulative_confirmed_imported_cases,"Santa Fe, Argentina","Santa Fe, Argentina",SAUCE VIEJO,SFN,SAAV


## Scrape weather data (v1 - direct calls to wunderground API)

#### Goal:

We need max_temp, mean_temp, min_temp, dew_point, precipitation, and wind attributes for each airport location, for the whole week. 

#### Challenge:

Wunderground API does not allow users to query directly for a weekly historical summary, like the direct webpages do in v1. The API does allow daily requests, however. We may be able to build the weekly summary by calling daily summaries and building out the weekly summary table for each airport. 

At this point it seems to be a matter of testing:
1. Speed on v1 (html scraping) vs v2 (wunderground api daily weather requests to weekly summaries)
2. Accuracy differences in model when using daily weather vs weekly weather for each airport.

In [ ]:
import requests
from collections import OrderedDict
import datetime

# Helper function: recursively calls itself to flatten dictionaries and lists.
# Needed to flatten the wunderground response for a single day's weather summary into a row in a dataframe
def flatten(json_object, container=None, name=''):
    if container is None:
        container = OrderedDict()
    if isinstance(json_object, dict):
        for key in json_object:
            flatten(json_object[key], container=container, name=name + key + '_')
    elif isinstance(json_object, list):
        for n, item in enumerate(json_object, 1):
            flatten(item, container=container, name=name + str(n) + '_')
    else:
        container[str(name[:-1])] = str(json_object)
    return container

# Helper function to create the wunderground API url, send request, transfer response to dataframe
def get_airport_dailyweather(wunderground_key, airport_code, yyyymmdd, cols_needed=None):
    # Create a url using passed params
    wunderground_url = 'http://api.wunderground.com/api/{0}/history_{1}/q/{2}.json'.format(wunderground_key, 
                                                                                       yyyymmdd, 
                                                                                       airport_code)
    r = requests.get(wunderground_url)
    df = pd.DataFrame(flatten(j['history']['dailysummary']), index=[0])
    cleaned_colnames = [col[2:] for col in df.columns]
    df.columns = cleaned_colnames
    # If the user passed a list of columns, filter down to only the ones needed
    if cols_needed:
        df = df[cols_needed].copy()
    return df

# Helper function to build list of weekly column names by max, min, mean suffixes
def generate_weekly_colnames(colnames_list, suffix_calcs_list=['max', 'mean', 'min']):
    weekly_colnames = []
    for colname in colnames_list:
        for suffix in suffix_calcs_list:
            weekly_colnames.append("{0}_{1}".format(colname, suffix))
    return weekly_colnames

# Helper function to create a list of weekly date strings formatted in YYYYMMDD from a single YYYYMMDD
def create_weekly_dates_list(yyyymmdd):
    dt = datetime.datetime.strptime(yyyymmdd, '%Y%m%d')
    week_dates = [ dt + datetime.timedelta(days=i) for i in range(7) ]
    weekly_dates_list = [dt.strftime('%Y%m%d') for dt in week_dates]
    return weekly_dates_list

In [ ]:
wunderground_key = "0a3a7926e3b32d4f"

In [ ]:
# These are the columns needed from each daily request to the wunderground api.
# Since there is no weekly historical return for each airport by using the API, but we can query each day, 
# These records will be generated for each day in the week and the weekly summary generated for the airport in Pandas

daily_cols_needed = ["date_pretty",
                     "date_tzname",
                     "date_year",
                     "date_mon",
                     "date_mday",
                     
                     "maxtempi",
                     "meantempi",
                     "mintempi",   
                     
                     "heatingdegreedays",
                     "coolingdegreedays",
                     "gdegreedays",
                     
                     "maxdewpti",
                     "meandewpti",
                     "mindewpti",
                     
                     "precipi",
                     "snowdepthi",
                     
                     "maxwspdi",
                     "meanwindspdi",
                     "minwspdi",
                     
#                      "wgusti_max",  # Wind Gust data not found in the API request
#                      "wgusti_ave",  # Wind Gust data not found in the API request
#                      "wgusti_min",  # Wind Gust data not found in the API request
                     
                     "maxpressurei",
                     "meanpressurei",
                     "minpressurei"
                    ]

##### Needed output
https://www.wunderground.com/history/airport/SABE/2017/1/12/WeeklyHistory.html
Out[51]:
Measurement                       
Max Temperature                Max       87 °F
                               Avg       84 °F
                               Min       79 °F
Mean Temperature               Max       80 °F
                               Avg       76 °F
                               Min       72 °F
Min Temperature                Max       71 °F
                               Avg       69 °F
                               Min       64 °F
Heating Degree Days (base 65)  Max           0
                               Avg           0
                               Min           0
                               Sum           0
Cooling Degree Days (base 65)  Max          14
                               Avg          11
                               Min           6
                               Sum          79
Growing Degree Days (base 50)  Max          30
                               Avg          26
                               Min          22
                               Sum         185
Dew Point                      Max       75 °F
                               Avg       64 °F
                               Min       48 °F
Precipitation                  Max     0.91 in
                               Avg     0.14 in
                               Min     0.00 in
                               Sum     0.95 in
Snowdepth                      Max           -
                               Avg           -
                               Min           -
                               Sum           -
Wind                           Max      36 mph
                               Avg      10 mph
                               Min       0 mph
Gust Wind                      Max      69 mph
                               Avg      42 mph
                               Min      25 mph
Sea Level Pressure             Max    30.01 in
                               Avg    29.81 in
                               Min    29.53 in
dtype: object

In [ ]:
weekly_cols_needed = generate_weekly_colnames(daily_cols_needed)
weekly_cols_needed

In [ ]:
test_date = "20170112"
test_airport = "SABE"

test_dailyweather_df = get_airport_dailyweather(wunderground_key, test_airport, test_date, cols_needed=daily_cols_needed)
# test_dailyweather_df = get_airport_dailyweather(wunderground_key, test_airport, test_date)
test_dailyweather_df

### Iteration Loop

In [ ]:
# Establish table of airport records with dates to pass to the helper functions
from datetime import timedelta

weather_scrape = (merge_all[['date','IATA','ICAO']]
                  .drop_duplicates()
                  .set_index(['IATA','ICAO']))

weather_scrape = (weather_scrape
                  .stack()
                  .reset_index(level=-1, drop=True)
                  .reset_index()
                  .rename(columns={0:'date'})
                  .dropna(subset=['IATA','ICAO'], how='all')
                 )

weather_scrape.head()

In [ ]:
len(weather_scrape)

In [ ]:
weather_dfs = []

for index, row in weather_scrape.iterrows():
    airport_code = row['IATA']
    yyyymmdd = row['date'].strftime('%Y%m%d')
    
    print("Retrieving weather for {0} on {1}...".format(airport_code, yyyymmdd))
    
    dailyweather_df = get_airport_dailyweather(wunderground_key, airport_code, yyyymmdd)
    
    weather_dfs.append(dailyweather_df)

In [ ]:
len(weather_dfs)

In [ ]:
weather_df = pd.concat(weather_dfs, axis=0).reset_index(drop=True)
weather_df

##### TODOs

- Append Airport codes
- Iterate on full dates (current daily records plus record for entire week)(test weather this is feasible/value-add)
- Build function to calc weekly weather metrics
- Compare against v1
- If weekly v2 does not work, test model performance against daily weather instead of weekly weather

Update: API call restrictions result in this method not being used for now. 

### Findings

Significant limitations in the wunderground API were found:

-	Calls Per Day and Calls Per Minute limitations (unless we pay for an upgraded plan)
-	No weekly history data available. Daily history requests can be made and compiled into a weekly history record for each airport, but that would result in about 120K calls to the API.

Reverting to the original approach to download weather data, which involves scraping of weekly history HTMLs in the following section.

## Scrape weather data (v2 - html downloaded tables - as originally performed)

In [49]:
from datetime import timedelta

weather_scrape = (merge_all[['date','IATA','ICAO']]
                  .drop_duplicates()
                  .set_index(['IATA','ICAO']))

weather_scrape['date1'] = weather_scrape.date - timedelta(days=7)
weather_scrape['date2'] = weather_scrape.date - timedelta(days=14)

weather_scrape = (weather_scrape
                  .stack()
                  .reset_index(level=-1, drop=True)
                  .reset_index()
                  .rename(columns={0:'date'})
                  .dropna(subset=['IATA','ICAO'], how='all')
                 )

weather_scrape.head()

,IATA,ICAO,date
0,AEP,SABE,2017-01-12
1,AEP,SABE,2017-01-05
2,AEP,SABE,2016-12-29
3,COR,SACO,2017-01-12
4,COR,SACO,2017-01-05


In [50]:
len(weather_scrape)

39978

In [73]:
def scrape_weekly_weather(date, df_row):
    # Scrape the weekly data table
    url_fmt = 'https://www.wunderground.com/history/airport/{}/{}/{}/{}/WeeklyHistory.html'
    try:
        url = url_fmt.format(df_row.ICAO, date.year, 
                             date.month, date.day)
        print(url)
    
    except:
        url = url_fmt.format(df_row.IATA, date.year, 
                             date.month, date.day)
        print("EEE"+url)
    
    try:
        table = pd.read_html(url)[0].dropna(subset=['Max','Avg','Min','Sum'], how='all')
        table.columns = ['Measurement','Max','Avg','Min','Sum']
        table.set_index('Measurement', inplace=True)
        table = table.stack()
        record = table.to_frame().T
        record['date'] = df_row.date
        record['ICAO'] = df_row.ICAO
        record['IATA'] = df_row.IATA
#         time.sleep(1.0)  # Why is the time.sleep(1.0) needed here? 
        return record

    except Exception as e:
        print(e)
        print('ERRRORRR')
#         table = pd.Series({'NULL':np.NaN}, index=pd.Index([0]))
    
#     return record

In [74]:
# Test loop
for index, row in weather_scrape[:5].iterrows():
    date = row['date']
    scrape_weekly_weather(date, row)

https://www.wunderground.com/history/airport/SABE/2017/1/12/WeeklyHistory.html
https://www.wunderground.com/history/airport/SABE/2017/1/5/WeeklyHistory.html
https://www.wunderground.com/history/airport/SABE/2016/12/29/WeeklyHistory.html
https://www.wunderground.com/history/airport/SACO/2017/1/12/WeeklyHistory.html
https://www.wunderground.com/history/airport/SACO/2017/1/5/WeeklyHistory.html


In [76]:
# Test Run
date = weather_scrape['date'].iloc[0]
df_row = weather_scrape.iloc[0]
scrape_weekly_weather(date, df_row)

https://www.wunderground.com/history/airport/SABE/2017/1/12/WeeklyHistory.html


Measurement Max Temperature               Mean Temperature                \
                        Max    Avg    Min              Max    Avg    Min   
0                     87 °F  84 °F  79 °F            80 °F  76 °F  72 °F   

Measurement Min Temperature               Heating Degree Days (base 65) ...   \
                        Max    Avg    Min                           Max ...    
0                     71 °F  69 °F  64 °F                             0 ...    

Measurement   Wind Gust Wind                 Sea Level Pressure            \
               Min       Max     Avg     Min                Max       Avg   
0            0 mph    69 mph  42 mph  25 mph           30.01 in  29.81 in   

Measurement                 date  ICAO IATA  
                  Min                        
0            29.53 in 2017-01-12  SABE  AEP  

[1 rows x 44 columns]

#### Weather Scraping Iteration

The problem with html scraping from wunderground is two-fold: 
1. It takes a long time.
2. If performed on a single system, the loop eventually crashes. (Still figuring out why)

For now, the focus is on retrieving a single pkl of the entire data, even if it takes a long time for a single system to perform this. Therefore, the attempted solution is to establish an iteration for the 40K records that buffers every 500 calls to a local file. Then once finished, goes back and assembles the final dataframe of data to use in the model. This can leverage most of the work that was already in place.

Process:

- Iterate on all .pkl files in the weather data folder

- Create a df list from each

- Concatenate all dfs into single weather dataframe

- pkl the single weather dataframe

In [77]:
len(weather_scrape)

39978

In [78]:
# Segment out the weather_scrape dataframe by index counts
def partition_df(df, partition_size):
    # Returns a list of dataframes from the original dataframe, each with a record count defined by partition_size
    segment_count = round(len(weather_scrape) / partition_size)
    partitions_list = np.array_split(weather_scrape, segment_count)
    return partitions_list

In [79]:
partitioned_weather_list = partition_df(weather_scrape, 500)

In [80]:
len(partitioned_weather_list)

80

In [81]:
partitioned_weather_list[0].shape

(500, 3)

In [132]:
# Create folder at root for the temp weather data
if not os.path.exists(root + '/scrape_weekly_weather_data'):
    os.makedirs(root + '/scrape_weekly_weather_data')

In [ ]:
# Iterate on each dataframe in the partitioned_weather_list, run weekly_weather scraper, download to pkl
for ix, weather_partition_df in enumerate(partitioned_weather_list):
    
    print("Scraping weather for partition {0} of {1}...".format(str(ix), len(partitioned_weather_list)))
    
    df_list = list()

    for index, row in weather_partition_df.iterrows():
        #TODO - Change to progress bar

        date = row['date']

        try:
            df = scrape_weekly_weather(date, row)
        except:
            df = pd.Series({'NULL':np.NaN}, index=pd.Index([row]))

        df_list.append((date, row.name, df))

    with open(root + '/scrape_weekly_weather_data/df_list_{}.pkl'.format(ix),'wb') as fh:
        dill.dump(df_list, fh)

In [108]:
weather_pkl_files = glob(root +'/scrape_weekly_weather_data/df_list_*.pkl')

In [91]:
weather_df_list = []
for file in weather_pkl_files:
    pickle = pd.read_pickle(file)
    weather_df_list.append(pickle[0][2])  # Dataframe is found at pickle[0][2]
    break

In [124]:
# read pkl files into pandas
weather_df_list = [pd.read_pickle(weather_pkl)[0][2] for weather_pkl in weather_pkl_files]  # Dataframe is found at pickle[0][2]
# concatenate them together
weather_df = pd.concat(weather_df_list)
weather_df.set_index(['ICAO', 'IATA', 'date'], inplace=True)

In [111]:
len(weather_df_list)

80

In [125]:
weather_df

Measurement          Cooling Degree Days (base 65)                Dew Point  \
                                               Avg  Max  Min  Sum       Avg   
ICAO IATA date                                                                
SABE AEP  2017-01-12                            11   14    6   79     64 °F   
SANT TUC  2017-03-13                             6   13    2   41     63 °F   
SBMT N/A  2016-06-25                             0    2    0    2     52 °F   
SBJP JPA  2016-08-27                            11   12   10   79     67 °F   
          2016-10-29                            14   17   12  100     69 °F   
SBBV BVB  2016-12-17                            20   22   17  139     71 °F   
SKPP PPN  2016-04-07                           NaN  NaN  NaN    0     61 °F   
SKSP ADZ  2016-05-26                            19   20   17  134     79 °F   
SKAR AXM  2016-06-23                             8    9    7   57     65 °F   
SKPS PSO  2016-10-29                             0    2    0    3     60 °F   
SARS N/A  2017-04-17                           NaN  NaN  NaN    0     62 °F   
SKUC AUC  2017-02-04                            16   19   14  115     71 °F   
SKBO BOG  2017-03-11                             0    0    0    0     50 °F   
          2017-05-06                             0    0    0    0     52 °F   
SKEJ EJA  2017-07-08                            19   20   17  130     75 °F   
SKSP ADZ  2017-07-29                            18   22    5  124     78 °F   
SKLC N/A  2016-01-23                            18   18   17  125     73 °F   
SKVV VVC  2016-02-27                            18   20   14  124     70 °F   
SKOC OCV  2016-03-26                           NaN  NaN  NaN  NaN         -   
SAZN NQN  2016-05-30                             0    0    0    0     43 °F   
SKSA RVE  2016-04-23                            17   22   10  120     75 °F   
SKPP PPN  2016-04-30                           NaN  NaN  NaN    0     63 °F   
SKSM SMR  2016-07-02                            22   24   19  153     76 °F   
SKUC AUC  2016-07-30                            14   16   12   97     76 °F   
SKLC N/A  2016-10-08                            18   20   16  124     76 °F   
MDAB N/A  2016-03-12                            13   16   12   94     75 °F   
SASA SLA  2016-06-25                             0    0    0    0     40 °F   
MDAB N/A  2016-10-22                            18   20   14  128     79 °F   
MDLR LRM  2017-03-04                            12   12   10   81     71 °F   
SEAM ATF  2016-12-14                             0    0    0    0     53 °F   
...                                            ...  ...  ...  ...       ...   
MMCZ CZM  2016-04-23                             9   12    8   66     70 °F   
MMZC ZCL  2016-05-21                             0    1    0    3     34 °F   
MMMV LOV  2016-07-09                            27   28   26  189     63 °F   
MMMD MID  2016-08-20                            20   22   16  140     76 °F   
MMQT QRO  2016-09-17                             2    6    0   14     55 °F   
MMZC ZCL  2016-11-05                             0    0    0    0     40 °F   
MMMV LOV  2016-12-24                             0    2    0    2     41 °F   
MMVR VER  2017-01-14                             5   10    0   34     62 °F   
MMQT QRO  2017-03-11                             1    4    0    9     46 °F   
SASA SLA  2016-11-02                             2    8    0   13     39 °F   
MMTG TGZ  2017-05-20                            21   24   14  146     67 °F   
MMCV CVM  2017-07-08                            18   22   12  126     67 °F   
MMOX OAX  2017-08-12                             6   10    1   39     58 °F   
MPMG PAC  2016-09-26                            16   17   14  113     76 °F   
MPDA DAV  2017-04-30                            16   17   15  111     75 °F   
KBTR BTR  2016-02-17                             0    1    0    1     47 °F   
KHTL HTL  2016-04-06                             0

In [126]:
full_weather_pkl = root + '\\weather_data.pkl'
weather_df.to_pickle(full_weather_pkl)

In [127]:
full_weather_pkl

'D:\\GitHub\\zika\\weather_data.pkl'

In [129]:
weather_df = pd.read_pickle(full_weather_pkl)
weather_df.head()

Measurement          Cooling Degree Days (base 65)              Dew Point  \
                                               Avg Max Min  Sum       Avg   
ICAO IATA date                                                              
SABE AEP  2017-01-12                            11  14   6   79     64 °F   
SANT TUC  2017-03-13                             6  13   2   41     63 °F   
SBMT N/A  2016-06-25                             0   2   0    2     52 °F   
SBJP JPA  2016-08-27                            11  12  10   79     67 °F   
          2016-10-29                            14  17  12  100     69 °F   

Measurement                        Growing Degree Days (base 50)          \
                        Max    Min                           Avg Max Min   
ICAO IATA date                                                             
SABE AEP  2017-01-12  75 °F  48 °F                            26  30  22   
SANT TUC  2017-03-13  72 °F  52 °F                            21  28  16   
SBMT N/A  2016-06-25  55 °F  46 °F                            12  17   7   
SBJP JPA  2016-08-27  72 °F  63 °F                            27  28  26   
          2016-10-29  72 °F  64 °F                            30  32  28   

Measurement           ...   Sea Level Pressure                     Snowdepth  \
                      ...                  Avg       Max       Min       Avg   
ICAO IATA date        ...                                                      
SABE AEP  2017-01-12  ...             29.81 in  30.01 in  29.53 in         -   
SANT TUC  2017-03-13  ...             30.02 in  30.27 in  29.55 in         -   
SBMT N/A  2016-06-25  ...             30.26 in  30.36 in  30.12 in         -   
SBJP JPA  2016-08-27  ...             30.01 in  30.09 in  29.92 in         -   
          2016-10-29  ...             29.94 in  30.01 in  29.83 in         -   

Measurement                         Wind                 
                     Max Min Sum     Avg     Max    Min  
ICAO IATA date                                           
SABE AEP  2017-01-12   -   -   -  10 mph  36 mph  0 mph  
SANT TUC  2017-03-13   -   -   -   6 mph  23 mph  0 mph  
SBMT N/A  2016-06-25   -   -   -   7 mph  17 mph  0 mph  
SBJP JPA  2016-08-27   -   -   -   9 mph  21 mph  2 mph  
          2016-10-29   -   -   -   9 mph  20 mph  4 mph  

[5 rows x 41 columns]

## Weather Data Retrieval V3? https://www.ncdc.noaa.gov/data-access/land-based-station-data/land-based-datasets/global-historical-climatology-network-ghcn

Pending check on version 2

## Mosquito sightings

In [ ]:
aedesitems = gis.content.search('Aedes aegypti', 'feature layer', outside_org=True, max_items=3)
for item in aedesitems:
    display(item)

In [ ]:
aedessightings = aedesitems[1]

In [ ]:
aedes = aedessightings.layers[0]

In [ ]:
aedesdf = aedes.query().df

In [ ]:
aedesdf.columns

In [ ]:
aedesdf.head(1)

In [ ]:
len(aedesdf)

In [ ]:
[x for x in zikadf.country.unique() if x not in aedesdf.COUNTRY.unique()]

In [ ]:
aedesdf = aedesdf.loc[aedesdf.YEAR >= 2006]

In [ ]:
mask = aedesdf.COUNTRY=='United States of America'
aedesdf.loc[mask,'COUNTRY'] = 'United States'

In [ ]:
spdf['X'] = spdf.SHAPE.apply(lambda x: x.x)
spdf['Y'] = spdf.SHAPE.apply(lambda x: x.y)
loc = spdf[['address', 'X', 'Y']]

In [ ]:
mosquito_coords = aedesdf[['Y', 'X']].values[np.newaxis, :]
places_coords = np.rollaxis(loc[['Y','X']].values[np.newaxis, :], 0, -1)
dist_coords = ((places_coords - mosquito_coords)**2).sum(axis=-1)
min_dist = dist_coords.min(axis=1)
mosquito_distance = loc[['address']].copy()
mosquito_distance['mosquito_dist'] = min_dist

In [ ]:
mosquito_distance.head()

In [ ]:
loc_mosquito_distance = root + '\\mosquito_distancet_df.pkl'
mosquito_distance.to_pickle(loc_mosquito_distance)

### Find nearest mosquito sightings from Zika locations (delete)

In [ ]:
def find_nearest_mosquitoes(location_filter, mosquito_filter):
    zikalyr.filter = location_filter
    aedes.filter = mosquito_filter
    
    nearest_airports = find_nearest(zikalyr, aedes, 'StraightLine', max_count=1,
                                   search_cutoff=100, search_cutoff_units='Miles')
    
    nearestdf = nearest_airports['connecting_lines_layer'].query().df
    
    return nearestdf[['From_Name', 'To_VECTOR', 'Total_Miles']]

In [ ]:
location_mosquito_df = pd.DataFrame()

In [ ]:
mosquito_filter = "Country = 'Colombia'"

location_filter = "Country = 'Colombia' AND OBJECTID < 1000"
df = find_nearest_mosquitoes(location_filter, airport_filter)

location_mosquito_df = location_mosquito_df.append(df, ignore_index=True)

In [ ]:
location_filter = "Country = 'Colombia' AND OBJECTID >= 1000"
df = find_nearest_mosquitoes(location_filter, airport_filter)
location_mosquito_df = location_mosquito_df.append(df, ignore_index=True)

In [ ]:
for country in zikadf.country.unique():
    print('Finding nearest mosquito sightings for locations in: ' + country)
    try:
        location_filter = "Country = '{}'".format(country)
        mosquito_filter = "Country = '{}'".format(country.upper())

        if country == 'United States':
            mosquito_filter = "Country = 'United States of America'"

        df = find_nearest_mosquitoes(location_filter, mosquito_filter)

        location_mosquito_df = location_mosquito_df.append(df, ignore_index=True)
    except:
        print('Unable to find nearest mosquito sightings for ' + country)

In [ ]:
location_mosquito_df.head()

## Population Density

In [ ]:
gis.content.search('title:"World Population Estimated Density 2015" owner:esri', 'Imagery layer', outside_org=True)[0]

In [ ]:
popdensity = gis.content.get('625e9da1afed40b78aaf412f519b22d3')
popdensity

In [ ]:
pop_density = popdensity.layers[0]

In [ ]:
pop_density.get_samples(zikadf['SHAPE'].iloc[0])

In [ ]:
len(zikadf)

In [ ]:
zikadf.head()

In [ ]:
zikadf['pop_density'] = np.nan

In [ ]:
df = zikadf

batch_size = pop_density.properties.maxRecordCount - 100
N = len(df)
geoms = []
for i in range(0, N, "Batch Geocoding.ipynb"_size):
    start = i
    stop = i + batch_size if i + batch_size < N else N
    points = [[pt.x, pt.y] for pt in df[start:stop].SHAPE]
    mpt = arcgis.geometry.Geometry({
        "points" : points,
        "spatialReference" : df.sr
    })
    
    res = pop_density.get_samples(mpt)
    for index in range(len(res)):
        df.ix[start + index, 'pop_density'] = res[index]['value']

In [ ]:
zikadf.head(1)

In [ ]:
loc_popdensity = root + '\\popdensity_df.pkl'
zikadf.to_pickle(loc_popdensity)

## GDP per capita

In [ ]:
gdp_pc =  gis.content.search('Gross Domestic Product Per Capita, 1960-2016, From World Bank', outside_org=True)[0]
gdp_pc

In [ ]:
gdp_lyr = gdp_pc.layers[0]

In [ ]:
gdp_df = gdp_lyr.query().df

In [ ]:
gdp_df.head()

## Class Balancing

In [ ]:
infection_data.head()

In [ ]:
infection_data.dtypes

In [ ]:
infection_sum = (infection_data[['date','location','zika_cases']]
                 .groupby(['date','location'], as_index=False)
                 .sum())

In [ ]:
infection_sum.head()

In [ ]:
def feasibility(df):
    cases_first_date = df.loc[df.date==df.date.min(), 'zika_cases'].values[0]
    date_first_date  = df.date.min()
    
    cases_max   = df.zika_cases.max()
    date_max    = df.loc[df.zika_cases==df.zika_cases.max(), 'date'].values[0]
    
    cases_last  = df.loc[df.date==df.date.max(), 'zika_cases'].values[0]
    date_last   = df.date.max()
    
    cases_total = df.zika_cases.sum()
    
    df2 = df.loc[df.zika_cases>0]
    
    if df2.shape[0]>=1:
        cases_first_nonzero = df2.loc[df2.date==df2.date.min(),'zika_cases'].values[0]
        date_first_nonzero  = df2.date.min()
    else:
        cases_first_nonzero = np.NaN
        date_first_nonzero = np.NaN
        
    #print(type(date_first_date), type(date_max), type(date_last), type(date_first_nonzero))
        
    return pd.Series({'cases_first_date' : cases_first_date,
                      'date_first_date'  : date_first_date,
                      'cases_first_nonzero' : cases_first_nonzero,
                      'date_first_nonzero'  : date_first_nonzero,
                      'cases_max'  : cases_max,
                      'date_max'   : date_max,
                      'cases_last' : cases_last,
                      'date_last'  : date_last,
                      'cases_total': cases_total})

In [ ]:
framework_key = (infection_sum
                   .groupby('location')
                   .apply(feasibility))

In [ ]:
framework_key.head(1)

In [ ]:
# Total size
print(framework_key.shape[0])

# Completely zero entries
print(framework_key.query('cases_max==0').shape[0])

# No zeros at all
print(framework_key.query('cases_first_date>0').shape[0])

# Number of entries that start at zero and have cases
print( ((framework_key.cases_max>0) & (framework_key.cases_first_date==0)).sum())

In [ ]:
framework_key['date_max'] = pd.to_datetime(framework_key.date_max)
framework_key['date_last'] = pd.to_datetime(framework_key.date_last)
framework_key['date_first_date'] = pd.to_datetime(framework_key.date_first_date)
framework_key['date_first_nonzero'] = pd.to_datetime(framework_key.date_first_nonzero)

In [ ]:
framework_key.dtypes

In [ ]:
mask = framework_key.cases_max > 0

framework_a_first = pd.concat([(framework_key
                                .loc[mask, ['date_first_nonzero']]
                                .assign(zika_bool=1)
                                .rename(columns={'date_first_nonzero':'date'})),
                               
                                # zero case data are taken from first date
                               (framework_key
                                .loc[mask.pipe(np.invert), ['date_first_date']]
                                .assign(zika_bool=0)
                                .rename(columns={'date_first_date':'date'}))]).sort_index().reset_index()


framework_a_max  = pd.concat([(framework_key
                                .loc[mask, ['date_max']]
                                .assign(zika_bool=1)
                                .rename(columns={'date_max':'date'})),
                              
                                # zero case data are taken from first date
                               (framework_key
                                .loc[mask.pipe(np.invert), ['date_first_date']]
                                .assign(zika_bool=0)
                                .rename(columns={'date_first_date':'date'}))]).sort_index().reset_index()

In [ ]:
framework_key.head()

In [ ]:
mask = framework_key.cases_max > 0

fwf = pd.concat([(framework_key
                                .loc[mask, ['date_first_nonzero','cases_first_date','cases_total']]
                                .assign(zika_bool=1)
                                .rename(columns={'date_first_nonzero':'date'})),
                               
                                # zero case data are taken from first date
                               (framework_key
                                .loc[mask.pipe(np.invert), ['date_first_date']]
                                .assign(zika_bool=0)
                                .rename(columns={'date_first_date':'date'}))]).sort_index().reset_index()

fwf['cases_first_date'] = fwf.cases_first_date.fillna(0).astype(np.int)
fwf['cases_total'] = fwf.cases_total.fillna(0).astype(np.int)

fwf.to_pickle(root+'/10_class_balancing_fwf.pkl')

In [ ]:
framework_a_first.head(1).T

In [ ]:
fwf.head(1).T

In [ ]:
framework_a_first.zika_bool.value_counts()

In [ ]:
framework_a_max.zika_bool.value_counts()

## Feature Engineering